# MovieLens データセットに対する協調フィルタリング

## 学習目標
1. BigQuery を使用してデータを探索する方法を理解する
2. モデルを使用して特定のユーザーに対するレコメンデーションを行う方法を理解する
3. モデルを使用してユーザーグループに対して特定のアイテムをレコメンドする方法を理解する

###### このノートブックは、Lakshmanan および Tigani 著の [BigQuery: The Definitive Guide](https://www.oreilly.com/library/view/google-bigquery-the/9781492044451/ "http://shop.oreilly.com/product/0636920207399.do") 第 9 章の一部に基づいています。
### MovieLens データセット
レコメンダー システムの実際の動作を説明するために、MovieLens データセットを使用します。これは、米国国立科学財団（NSF）の資金提供を受けて、ミネソタ大学コンピュータサイエンス・工学部の研究室である GroupLens によって公開された映画レビューのデータセットです。

データをダウンロードし、次を使用して BigQuery テーブルとして読み込みます。

In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]

%env PROJECT=$PROJECT

%load_ext google.cloud.bigquery

In [ ]:
%%bash
rm -r bqml_data
mkdir bqml_data
cd bqml_data
curl -O 'http://files.grouplens.org/datasets/movielens/ml-20m.zip'
unzip ml-20m.zip
yes | bq rm -r $PROJECT:movielens
bq --location=US mk --dataset \
    --description 'Movie Recommendations' \
    $PROJECT:movielens
bq --location=US load --source_format=CSV \
    --autodetect movielens.ratings ml-20m/ratings.csv
bq --location=US load  --source_format=CSV \
    --autodetect movielens.movies_raw ml-20m/movies.csv

## データの探索
これで、<a href="https://console.cloud.google.com/bigquery">BigQuery</a> で 2 つのテーブルが利用可能になっているはずです。

協調フィルタリングは、ユーザーに対する商品のレコメンデーション、または商品に対するユーザーのターゲティングを行う手法を提供します。出発点となるのは、ユーザー ID、アイテム ID、ユーザーが商品に付けた評価の 3 つの列を持つテーブル <b>movielens.ratings</b> です。このテーブルはスパース（疎）であっても構いません（ユーザーがすべての商品を評価する必要はありません）。そして、評価のみに基づいて類似したユーザーや類似した商品を検出し、まだ見たことのない商品に対してユーザーが付けるであろう評価を予測します。これにより、予測評価が最も高い商品をユーザーにレコメンドしたり、予測評価が最も高いユーザーに対して商品をターゲティングしたりできます。

In [ ]:
%%bigquery --project $PROJECT
SELECT *
FROM movielens.ratings
LIMIT 10

簡単な探索クエリを実行すると、データセットが 13 万 8,000 人以上のユーザー、約 2 万 7,000 本の映画、2,000 万件強の評価で構成されていることがわかり、データが正常に読み込まれたことが確認できます。

In [ ]:
%%bigquery --project $PROJECT
SELECT 
    COUNT(DISTINCT userId) numUsers,
    COUNT(DISTINCT movieId) numMovies,
    COUNT(*) totalRatings
FROM movielens.ratings

次のクエリを使用して最初の数本の映画を調べてみると、genres 列がフォーマットされた文字列になっていることがわかります。

In [ ]:
%%bigquery --project $PROJECT
SELECT *
FROM movielens.movies_raw
WHERE movieId < 5

ジャンルを配列にパースし、次のようにテーブルを書き換えることができます。

In [ ]:
%%bigquery --project $PROJECT
CREATE OR REPLACE TABLE movielens.movies AS
    SELECT * REPLACE(SPLIT(genres, "|") AS genres)
    FROM movielens.movies_raw

In [ ]:
%%bigquery --project $PROJECT
SELECT *
FROM movielens.movies
WHERE movieId < 5

## 行列分解（Matrix Factorization）
行列分解は、評価行列をユーザー因子（user factors）とアイテム因子（item factors）と呼ばれる 2 つの低次元ベクトルに分解する協調フィルタリング手法です。ユーザー因子は user_id の低次元表現であり、アイテム因子も同様に item_id を表現します。

**注: 行列分解（MF）モデルのトレーニングには BigQuery の定額料金（フラットレート）契約が必要です。そのため、ここでは外部プロジェクトから事前トレーニング済みモデルを取得します。**

BigQuery で定額料金を有効にしている場合は、次のクエリを使用して MF モデルをトレーニングできます。
```SQL

CREATE OR REPLACE MODEL movielens.recommender
options(model_type='matrix_factorization',
        user_col='userId', item_col='movieId', rating_col='rating')
AS

SELECT 
userId, movieId, rating
FROM movielens.ratings

```

In [ ]:
%%bigquery --project $PROJECT
SELECT iteration, loss, duration_ms
-- 注: 独自のモデルを使用している場合は cloud-training-demos を削除してください:
FROM ML.TRAINING_INFO(MODEL `cloud-training-demos.movielens.recommender`)


## レコメンデーションの実行

トレーニング済みモデルを使用して、レコメンデーションを提供できるようになりました。たとえば、userId が 903 のユーザーにレコメンドする最適なコメディ映画を見つけてみましょう。以下のクエリでは、トレーニング済みのレコメンデーション モデルを渡して ML.PREDICT を呼び出し、予測を実行する movieId と userId のセットを指定しています。この例では、userId は 1 つ（903）のみですが、ジャンルに「Comedy」が含まれるすべての映画を対象としています。

In [ ]:
%%bigquery --project $PROJECT
SELECT * FROM
ML.PREDICT(MODEL `cloud-training-demos.movielens.recommender`, (
  SELECT 
    movieId, title, 903 AS userId
  FROM movielens.movies, UNNEST(genres) g
  WHERE g = 'Comedy'
))
ORDER BY predicted_rating DESC
LIMIT 5

## すでに評価済みの映画の除外
当然ながら、これにはユーザーが過去に視聴して評価した映画も含まれます。それらを除外しましょう。

**TODO 1**: すでに視聴済みの映画を含まない、ユーザー 903 向けの予測を作成します。

In [ ]:
%%bigquery --project $PROJECT
SELECT * FROM
ML.PREDICT(MODEL `cloud-training-demos.movielens.recommender`, (
  WITH seen AS (
    SELECT ARRAY_AGG(movieId) AS movies 
    FROM movielens.ratings
    WHERE userId = 903
  )
  SELECT 
    movieId, title, 903 AS userId
  FROM movielens.movies, UNNEST(genres) g, seen
  WHERE g = 'Comedy' AND movieId NOT IN UNNEST(seen.movies)
))
ORDER BY predicted_rating DESC
LIMIT 5

このユーザーの場合、偶然にも同じ映画のセットが得られました。予測評価の上位には、ユーザーがすでに視聴した映画は含まれていませんでした。

## 顧客ターゲティング

前のセクションでは、特定のユーザーに対して高評価が得られる映画を特定する方法を見ました。場合によっては、特定の商品があり、それを評価してくれそうな顧客を見つける必要があります。たとえば、評価が 1 件しかない movieId = 96481（『American Mullet』）について、さらにレビューを集めたいと考え、最も高い評価を付けてくれそうな 5 人のユーザーにクーポンを送りたいとします。 


**TODO 2**: 『*American Mullet (2001)*』を楽しんでくれそうな上位 5 人のユーザーを見つけます。

In [ ]:
%%bigquery --project $PROJECT
SELECT * FROM
ML.PREDICT(MODEL `cloud-training-demos.movielens.recommender`, (
  WITH allUsers AS (
     SELECT DISTINCT userId
     FROM movielens.ratings
  )
  SELECT 
    96481 AS movieId, 
    (SELECT title FROM movielens.movies WHERE movieId=96481) title,
    userId
  FROM
    allUsers
))
ORDER BY predicted_rating DESC
LIMIT 5

### すべてのユーザーと映画に対するバッチ予測
すべてのユーザーと映画の組み合わせに対して予測を実行したい場合はどうすればよいでしょうか？前のクエリのように個別のユーザーや映画を抽出する代わりに、トレーニング中に検出されたすべての movieId と userId に対してバッチ予測を実行する便利な関数が提供されています。ここでは LIMIT を適用しています。そうしないと、すべてのユーザーと映画の予測が返され、ノートブックがクラッシュしてしまうためです。

In [ ]:
%%bigquery --project $PROJECT
SELECT *
FROM ML.RECOMMEND(MODEL `cloud-training-demos.movielens.recommender`)
LIMIT 10

上のセクションで見たように、ユーザーが過去に視聴して評価した映画を除外することは可能です。視聴済みの映画がデフォルトで除外されない理由は、過去にユーザーが気に入ったレストランを再度レコメンドすることが想定されるような状況（たとえばレストランのレコメンデーションなど）が存在するためです。

Copyright 2019 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.